In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import scanpy as sc
import anndata

import glob

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgb
from pandas.api.types import CategoricalDtype
from scipy.stats import mannwhitneyu

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/hicluster'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
log2_min = np.log2(2500)        # ≈ 11.29
log2_max = np.log2(250000000) # ≈ 27.89
step = 0.125
log2_bins = np.arange(log2_min, log2_max + step, step)

bin_edges = 2 ** log2_bins  # convert back to basepair units
bin_edges.shape

In [ ]:
2e6

In [ ]:
cutoff = 2e6
short_bins = bin_edges[bin_edges < cutoff][1:]
long_bins = bin_edges[bin_edges > cutoff]

In [ ]:
contact_df = pd.read_csv(f'{projdir}/csv/distance/merged.csv.gz', sep='\t', header=0, index_col=0)
contact_df.shape

In [ ]:
contact_df.head()

In [ ]:
contact_df.columns = bin_edges[1:]

In [ ]:
short_bins

In [ ]:
short_sum = contact_df.loc[:,short_bins].sum(axis=1)
long_sum = contact_df.loc[:,long_bins].sum(axis=1)
sum_df = pd.DataFrame(zip(short_sum, long_sum))
sum_df.columns = ['short', 'long']
sum_df['short/long'] = sum_df['short'] / sum_df['long']
sum_df.index = contact_df.index

In [ ]:
sum_df.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
clusters = pd.read_csv(f'{indir}/csv/label_transfer/xgboost_time_v7.csv', sep='\t', index_col=0)
clusters.shape

In [ ]:
blacklist = ['Skin', 'Inflamed']
mask = [x not in blacklist for x in clusters['cluster']]
tmp_clusters = clusters[mask]
tmp_clusters.shape

In [ ]:
clusters.head()

In [ ]:
sum_df['cluster'] = tmp_clusters.reindex(sum_df.index)['cluster']
sum_df['cluster_sub'] = tmp_clusters.reindex(sum_df.index)['cluster_sub']

cat_type = CategoricalDtype(categories=tmp_order, ordered=True)
sum_df['cluster'] = sum_df['cluster'].astype(cat_type)
sum_df['donor'] = tmp_clusters.reindex(sum_df.index)['line']
sum_df.head()

In [ ]:
sum_df.groupby('cluster').mean()

In [ ]:
sum_df.shape

In [ ]:
sum_df.to_csv(f'{projdir}/csv/distance/short_long_meta.csv', sep='\t')

In [ ]:
fig, axes = plt.subplots(1, figsize=(7, 6))

x_lab = 'cluster'
y_lab = 'short'
hue = 'donor'

ax = sns.boxplot(data=sum_df, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('Fibro score', fontsize=8)
ax.set_ylabel('ESC score', fontsize=8)
ax.set_title('3C', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=8, labelrotation=45)
ax.tick_params(axis='y', labelsize=8, labelrotation=0)

# ax.set_xlim(0, 0.4)
# ax.set_ylim(0, 1.2)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
fig, axes = plt.subplots(1, figsize=(7, 6))

x_lab = 'cluster'
y_lab = 'short'
hue = 'donor'

ax = sns.boxplot(data=sum_df, x=x_lab, y=y_lab, showfliers=False)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Short range contacts %', fontsize=12)
ax.set_title('3C', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=8, labelrotation=45)
ax.tick_params(axis='y', labelsize=8, labelrotation=0)

# ax.set_xlim(0, 0.4)
# ax.set_ylim(0, 1.2)
# sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
fig, axes = plt.subplots(1, figsize=(7, 6))

x_lab = 'cluster'
y_lab = 'long'
hue = 'donor'

ax = sns.boxplot(data=sum_df, x=x_lab, y=y_lab, showfliers=False)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Long range contacts %', fontsize=12)
ax.set_title('3C', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=8, labelrotation=45)
ax.tick_params(axis='y', labelsize=8, labelrotation=0)

# ax.set_xlim(0, 0.4)
# ax.set_ylim(0, 1.2)
# sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
fig, axes = plt.subplots(1, figsize=(7, 6))

x_lab = 'cluster'
y_lab = 'short/long'
hue = 'donor'

ax = sns.boxplot(data=sum_df, x=x_lab, y=y_lab, showfliers=False)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Short/Long Ratio', fontsize=12)
ax.set_title('3C', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=8, labelrotation=45)
ax.tick_params(axis='y', labelsize=8, labelrotation=0)

# ax.set_xlim(0, 0.4)
# ax.set_ylim(0, 1.2)
# sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
to_plot = sum_df.melt(id_vars=['cluster'], value_vars=['short', 'long'])
fig, axes = plt.subplots(1, figsize=(7, 6))

x_lab = 'cluster'
y_lab = 'value'
hue = 'variable'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Contact %', fontsize=12)
ax.set_title('Contact length distribution', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=12, labelrotation=45)
ax.tick_params(axis='y', labelsize=12, labelrotation=0)

# ax.set_xlim(0, 0.4)
# ax.set_ylim(0, 1.2)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
plot_order = ['Start', 'Delayed', 'Fail1', 'Sendai', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
to_plot = sum_df.melt(id_vars=['cluster'], value_vars=['short', 'long'])
to_plot['tmp_cluster'] = to_plot['cluster'].copy()
cat_type = CategoricalDtype(categories=plot_order, ordered=True)
to_plot['tmp_cluster'] = to_plot['tmp_cluster'].astype(cat_type)

fig, axes = plt.subplots(1, figsize=(7, 6))

x_lab = 'tmp_cluster'
y_lab = 'value'
hue = 'variable'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Contact %', fontsize=12)
ax.set_title('Contact length distribution', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=12, labelrotation=45)
ax.tick_params(axis='y', labelsize=12, labelrotation=0)

# ax.set_xlim(0, 0.4)
# ax.set_ylim(0, 1.2)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


In [ ]:
outdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/final_figures/csv/figure3'
to_plot.to_csv(f'{outdir}/all_short_long_contacts.csv', sep='\t')

In [ ]:
whitelist = ['Start', 'Delayed', 'Fail1']
mask = [x in whitelist for x in to_plot['cluster']]
final_out = to_plot[mask]
final_out.head()

In [ ]:
outdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/final_figures/csv/figure3'
final_out.to_csv(f'{outdir}/fail1_short_long_contacts.csv', sep='\t')

In [ ]:
to_plot = sum_df.melt(id_vars=['cluster_sub'], value_vars=['short', 'long'])
whitelist = ['Inter1_1', 'Inter1_2', 'Delayed_1', 'Delayed_2', 'IPS_1', 'IPS_2']
mask = [x in whitelist for x in to_plot['cluster_sub']]
to_plot = to_plot[mask]
cat_type = CategoricalDtype(categories=whitelist, ordered=True)
to_plot['cluster_sub'] = to_plot['cluster_sub'].astype(cat_type)

fig, axes = plt.subplots(1, figsize=(7, 6))

x_lab = 'cluster_sub'
y_lab = 'value'
hue = 'variable'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Contact %', fontsize=12)
ax.set_title('Contact length distribution', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=12, labelrotation=45)
ax.tick_params(axis='y', labelsize=12, labelrotation=0)

# ax.set_xlim(0, 0.4)
# ax.set_ylim(0, 1.2)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
sum_df

In [ ]:
to_plot = sum_df.melt(id_vars=['cluster'], value_vars=['short/long'])
to_plot['tmp_cluster'] = to_plot['cluster'].copy()
cat_type = CategoricalDtype(categories=plot_order, ordered=True)
to_plot['tmp_cluster'] = to_plot['tmp_cluster'].astype(cat_type)

fig, axes = plt.subplots(1, figsize=(7, 6))

x_lab = 'tmp_cluster'
y_lab = 'value'
# hue = 'variable'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Contact %', fontsize=12)
ax.set_title('Contact length distribution', fontsize=20)
ax.grid(False)
ax.tick_params(axis='x', labelsize=12, labelrotation=45)
ax.tick_params(axis='y', labelsize=12, labelrotation=0)

# ax.set_xlim(0, 0.4)
# ax.set_ylim(0, 1.2)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


In [ ]:
to_plot.head()

In [ ]:
outdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/final_figures/csv/figure3'
to_plot.to_csv(f'{outdir}/clusters_short_long.csv', sep='\t')

In [ ]:
to_plot.head()

In [ ]:
h5ad_path = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/higashi/h5ad/embedding/higashi_filtered.h5ad'
adata = sc.read_h5ad(h5ad_path)

In [ ]:
adata

In [ ]:
adata.obs['short/long'] = sum_df.reindex(adata.obs.index)['short/long']

In [ ]:
adata.obs['cluster'] = tmp_clusters.reindex(adata.obs.index)['cluster']
adata.obs['cluster_sub'] = tmp_clusters.reindex(adata.obs.index)['cluster_sub']
tmp_adata = adata[~adata.obs['cluster'].isna()]

In [ ]:
sc.pl.umap(tmp_adata, color=['cluster', 'short/long'], vmin=0, vmax=3)

In [ ]:
sc.pl.umap(tmp_adata, color=['cluster_sub'])

In [ ]:
sc.pl.umap(tmp_adata, color=['cluster_sub'], groups=['Delayed_2'])

In [ ]:
s = 'Delayed_1'
tmp_meta = tmp_adata[tmp_adata.obs['cluster_sub'] == s]
tmp_meta.obs['time'].value_counts().sort_index().sum()

In [ ]:
s = 'Delayed_2'
tmp_meta = tmp_adata[tmp_adata.obs['cluster_sub'] == s]
tmp_meta.obs['time'].value_counts().sort_index().sum()

In [ ]:
fail_branch = ['Start', 'Delayed', 'Fail1']
for s in fail_branch:
    tmp_meta = tmp_adata[tmp_adata.obs['cluster'] == s]
    print(tmp_meta.obs['time'].value_counts().sort_index())

In [ ]:
fail_branch = ['Start', 'Delayed_1', 'Fail1']
for s in fail_branch:
    tmp_meta = tmp_adata[tmp_adata.obs['cluster_sub'] == s]
    print(tmp_meta.obs['time'].value_counts().sort_index())

In [ ]:
sc.pl.umap(tmp_adata, color=['cluster'], groups=['Delayed'])

In [ ]:
sc.pl.umap(tmp_adata, color=['cluster_sub'], groups=['Delayed_1'])

In [ ]:
sc.pl.umap(tmp_adata, color=['cluster_sub'], groups=['Delayed_2'])

In [ ]:
sc.pl.umap(tmp_adata, color=['cluster'], groups=['Start'])

In [ ]:
sc.pl.umap(tmp_adata, color=['cluster'], groups=['Fail1'])

In [ ]:
tmp_adata.obs['tmp_time'] = [f'D{x}' for x in tmp_adata.obs['time']]

In [ ]:
# sc.pl.umap(tmp_adata, color=['tmp_time'], groups=['D0'])
# sc.pl.umap(tmp_adata, color=['tmp_time'], groups=['D9'])
# sc.pl.umap(tmp_adata, color=['tmp_time'], groups=['D12'])
# sc.pl.umap(tmp_adata, color=['tmp_time'], groups=['D14'])
# sc.pl.umap(tmp_adata, color=['tmp_time'], groups=['D16'])
# sc.pl.umap(tmp_adata, color=['tmp_time'], groups=['D18'])

### let's compare short/long ratio

In [ ]:
whitelist = ['Start', 'Delayed', 'Fail1']

mask = [x in whitelist for x in sum_df['cluster']]
tmp_contacts = sum_df[mask]

In [ ]:
a =  sum_df[sum_df['cluster'] == 'Start']['short/long'].values
b = sum_df[sum_df['cluster'] == 'Delayed']['short/long'].values
mannwhitneyu(a, b)

In [ ]:
a =  sum_df[sum_df['cluster'] == 'Start']['short/long'].values
b = sum_df[sum_df['cluster'] == 'Fail1']['short/long'].values
mannwhitneyu(a, b)

In [ ]:
a =  sum_df[sum_df['cluster'] == 'Start']['short/long'].values
b = sum_df[sum_df['cluster'] == 'IPS']['short/long'].values
mannwhitneyu(a, b)

In [ ]:
!date